## 1. Settings

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Locate the repository from the current folder (repository root or code/).
working_folder = Path.cwd().resolve()
project_folder = next(
    (folder for folder in (working_folder, *working_folder.parents)
     if all((folder / name).is_dir() for name in ("code", "dataset_raw", "dataset_analysis"))),
    None,
)
if project_folder is None:
    raise FileNotFoundError(
        "Open Jupyter from the repository root or its code folder. "
        "Expected code/, dataset_raw/, and dataset_analysis/."
    )
raw_folder = project_folder / "dataset_raw"
output_folder = project_folder / "output"
data_folder = project_folder / "dataset_analysis"
data_folder.mkdir(parents=True, exist_ok=True)


## 2. Look at the original OECD columns and a few example rows

In [2]:
# Reading in chunks avoids loading the entire large CSV into memory.
insurance_file = raw_folder / "OECD_premiums_claims_expenses.csv.gz"
insurance_example = pd.read_csv(insurance_file, nrows=5, dtype=str, keep_default_na=False)
print("OECD example rows:")
display(insurance_example)

print("\nOriginal OECD columns:")
for column_number, column_name in enumerate(insurance_example.columns, start=1):
    print(f"{column_number:2}. {column_name}")

OECD example rows:


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,FREQ,Frequency of observation,MEASURE,Measure,...,OBS_STATUS,Observation status,CONF_STATUS,Confidentiality status,UNIT_MULT,Unit multiplier,CURRENCY,Currency,DECIMALS,Decimals
0,DATAFLOW,OECD.DAF.CM:DSD_INS@DF_UNDERWRITING(1.0),"Premiums, claims and expenses of insurance com...",I,USA,United States,A,Annual,GRS,Gross written premiums,...,A,Normal value,F,Free (free for publication),6,Millions,USD,US dollar,0,Zero
1,DATAFLOW,OECD.DAF.CM:DSD_INS@DF_UNDERWRITING(1.0),"Premiums, claims and expenses of insurance com...",I,USA,United States,A,Annual,GRS,Gross written premiums,...,A,Normal value,F,Free (free for publication),6,Millions,USD,US dollar,0,Zero
2,DATAFLOW,OECD.DAF.CM:DSD_INS@DF_UNDERWRITING(1.0),"Premiums, claims and expenses of insurance com...",I,USA,United States,A,Annual,GRS,Gross written premiums,...,A,Normal value,F,Free (free for publication),6,Millions,USD,US dollar,0,Zero
3,DATAFLOW,OECD.DAF.CM:DSD_INS@DF_UNDERWRITING(1.0),"Premiums, claims and expenses of insurance com...",I,USA,United States,A,Annual,GRS,Gross written premiums,...,A,Normal value,F,Free (free for publication),6,Millions,USD,US dollar,0,Zero
4,DATAFLOW,OECD.DAF.CM:DSD_INS@DF_UNDERWRITING(1.0),"Premiums, claims and expenses of insurance com...",I,USA,United States,A,Annual,GRS,Gross written premiums,...,A,Normal value,F,Free (free for publication),6,Millions,USD,US dollar,0,Zero



Original OECD columns:
 1. STRUCTURE
 2. STRUCTURE_ID
 3. STRUCTURE_NAME
 4. ACTION
 5. REF_AREA
 6. Reference area
 7. FREQ
 8. Frequency of observation
 9. MEASURE
10. Measure
11. UNIT_MEASURE
12. Unit of measure
13. PREMIUMS
14. Premiums
15. OWNERSHIP
16. Ownership
17. INSURANCE_TYPE
18. Insurance type
19. INSURANCE_BUSINESS
20. Insurance business
21. INSURER_TYPE
22. Insurer type
23. CONTRACT_TYPE
24. Contract type
25. EMPLOYER_TYPE
26. Employer type
27. RISK_LOCATION
28. Risk location
29. COUNTERPART_AREA
30. Counterpart area
31. INSURANCE_CLASS
32. Insurance class
33. DESTINATION
34. Destination
35. TIME_PERIOD
36. Time period
37. OBS_VALUE
38. Observation value
39. OBS_STATUS
40. Observation status
41. CONF_STATUS
42. Confidentiality status
43. UNIT_MULT
44. Unit multiplier
45. CURRENCY
46. Currency
47. DECIMALS
48. Decimals


## 3. Count missing cells and list distinct values across the full raw file

In [3]:
# A blank cell is missing. Zero is a reported value, not a missing value.
column_names = insurance_example.columns.tolist()
missing_cell_counts = pd.Series(0, index=column_names, dtype="int64")
distinct_column_values = {column: set() for column in column_names}
country_label_parts = []
total_rows = 0
numeric_observations = 0
reported_zeros = 0
reported_negatives = 0

# Keep the analysis rows while reading, so script 02 can use a small input file.
analysis_filters = {
    "FREQ": "A", "MEASURE": "GRS", "UNIT_MEASURE": "USD", "Ownership": "All undertakings", "CONTRACT_TYPE": "_T",
    "RISK_LOCATION": "_Z", "PREMIUMS": "_Z", "INSURER_TYPE": "_Z", "EMPLOYER_TYPE": "_Z", "COUNTERPART_AREA": "_Z",
    "INSURANCE_CLASS": "_Z", "DESTINATION": "_Z", "CURRENCY": "USD",
}
insurance_types = ["Life", "Composite: life", "Non-life", "Composite: non-life"]
business_types = ["Direct business", "Reinsurance accepted", "Total"]
analysis_parts = []

for chunk_number, insurance_chunk in enumerate(
    pd.read_csv(insurance_file, dtype=str, keep_default_na=False, chunksize=100_000),
    start=1,
):
    total_rows += len(insurance_chunk)
    missing_cell_counts += insurance_chunk.apply(lambda column: column.str.strip().eq("").sum())
    for column in column_names:
        distinct_column_values[column].update(insurance_chunk[column].unique())
    country_label_parts.append(insurance_chunk[["REF_AREA", "Reference area"]].drop_duplicates())

    premium_text = insurance_chunk["OBS_VALUE"].str.strip().str.lstrip("'")
    premium_text = premium_text.mask(premium_text.eq(""))
    
    premium_values = pd.to_numeric(premium_text, errors="raise")
    numeric_observations += premium_values.notna().sum()
    reported_zeros += premium_values.eq(0).sum()
    reported_negatives += premium_values.lt(0).sum()

    keep_rows = insurance_chunk["Insurance type"].isin(insurance_types)
    keep_rows &= insurance_chunk["Insurance business"].isin(business_types)
    
    for column, required_value in analysis_filters.items():
        keep_rows &= insurance_chunk[column].eq(required_value)
    
    # Retain pre-1995 history for imputation, even though the analysis starts in 1995.
    analysis_parts.append(insurance_chunk.loc[keep_rows].copy())

print(f"Finished processes")

Finished processes


## 4. Make a readable column-information table

In [4]:
column_information_rows = []
for column in column_names:
    nonblank_values = sorted(value for value in distinct_column_values[column] if value.strip())
    column_information_rows.append({
        "Column": column,
        "Loaded_type": "text (source values preserved)",
        "Missing_cells": int(missing_cell_counts[column]),
        "Missing_percent": 100 * missing_cell_counts[column] / total_rows,
        "Distinct_nonblank_values": len(nonblank_values),
        "Examples": " | ".join(nonblank_values[:5]),
    })

column_information = pd.DataFrame(column_information_rows)
display(column_information)




,Column,Loaded_type,Missing_cells,Missing_percent,Distinct_nonblank_values,Examples
0,STRUCTURE,text (source values preserved),0,0.00000,1,DATAFLOW
1,STRUCTURE_ID,text (source values preserved),0,0.00000,1,OECD.DAF.CM:DSD_INS@DF_UNDERWRITING(1.0)
2,STRUCTURE_NAME,text (source values preserved),0,0.00000,1,"Premiums, claims and expenses of insurance com..."
3,ACTION,text (source values preserved),0,0.00000,1,I
4,REF_AREA,text (source values preserved),0,0.00000,72,ARG | AUS | AUT | BEL | BGR
5,Reference area,text (source values preserved),0,0.00000,72,Argentina | Australia | Austria | Belgium | Be...
6,FREQ,text (source values preserved),0,0.00000,1,A
7,Frequency of observation,text (source values preserved),0,0.00000,1,Annual
8,MEASURE,text (source values preserved),0,0.00000,6,CED | CLM | COM | EXP | GRS
9,Measure,text (source values preserved),0,0.00000,6,Commissions | Gross claims payments | Gross op...


In [5]:
# Keep the meaning of the main analysis fields next to the code that reads them.
important_column_meanings = pd.DataFrame({
    "Column": ["REF_AREA", "Reference area", "TIME_PERIOD", "OBS_VALUE", "UNIT_MULT",
               "Insurance type", "Insurance business", "OBS_STATUS", "MEASURE", "Ownership"],
    "Meaning": ["Country/area code", "Country/area name", "Observation year",
                "Reported amount; blank is missing", "Power-of-ten multiplier; 6 means millions",
                "Life, Non-life, or the corresponding Composite type",
                "Direct business, accepted reinsurance, or reported Total",
                "Observation status; B marks a reported break",
                "Premium measure; GRS selects gross written premiums",
                "Reporting scope; this project selects All undertakings"],
})
display(important_column_meanings)

,Column,Meaning
0,REF_AREA,Country/area code
1,Reference area,Country/area name
2,TIME_PERIOD,Observation year
3,OBS_VALUE,Reported amount; blank is missing
4,UNIT_MULT,Power-of-ten multiplier; 6 means millions
5,Insurance type,"Life, Non-life, or the corresponding Composite..."
6,Insurance business,"Direct business, accepted reinsurance, or repo..."
7,OBS_STATUS,Observation status; B marks a reported break
8,MEASURE,Premium measure; GRS selects gross written pre...
9,Ownership,Reporting scope; this project selects All unde...


## 5. Check countries, years, and insurance categories

In [6]:
country_labels = pd.concat(country_label_parts).drop_duplicates()
country_labels = country_labels.rename(columns={"Reference area": "Country"})
country_labels = country_labels.loc[country_labels["REF_AREA"].ne("OECD")].sort_values("Country")
if country_labels["REF_AREA"].duplicated().any():
    raise ValueError("A country code has more than one label. Review the source.")

for column in ["TIME_PERIOD", "Insurance type", "Insurance business", "Measure", "Ownership", "UNIT_MULT"]:
    print(f"{column}: {sorted(distinct_column_values[column])}")

scoped_insurance_rows = pd.concat(analysis_parts, ignore_index=True)
scoped_insurance_rows = scoped_insurance_rows.loc[
    scoped_insurance_rows["REF_AREA"].isin(country_labels["REF_AREA"])
].copy()

print("")

display(scoped_insurance_rows.groupby(["Insurance type", "Insurance business"]).size().unstack(fill_value=0))

TIME_PERIOD: ['1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Insurance type: ['Composite', 'Composite: life', 'Composite: non-life', 'Life', 'Non-life', 'Total']
Insurance business: ['Direct business', 'Not applicable', 'Reinsurance accepted', 'Total']
Measure: ['Commissions', 'Gross claims payments', 'Gross operating expenses', 'Gross written premiums', 'Net written premiums', 'Premiums ceded']
Ownership: ['All undertakings', 'Branches and agencies of foreign undertakings', 'Domestic undertakings', 'Foreign controlled undertakings']
UNIT_MULT: ['6']



Insurance business,Direct business,Reinsurance accepted,Total
Insurance type,,,
Composite: life,1183,1146,1148
Composite: non-life,1183,1146,1148
Life,1518,1429,1493
Non-life,1553,1472,1502


## 6. Inspect the World Bank inputs

In [7]:
worldbank_overview_rows = []
for filename in ["worldbank_gdp.csv", "worldbank_population_total.csv"]:
    worldbank_data = pd.read_csv(raw_folder / filename, dtype=str, keep_default_na=False)
    year_columns = [str(year) for year in range(1995, 2025)]
    yearly_values = worldbank_data[year_columns].replace("", float("nan"))
    print(f"\n{filename}")
    display(worldbank_data.head())
    print("Indicators:", worldbank_data["Indicator Name"].unique().tolist())
    worldbank_overview_rows.append({
        "File": filename, "Rows": len(worldbank_data), "Columns": len(worldbank_data.columns),
        "Missing_year_cells_1995_2024": int(yearly_values.isna().sum().sum()),
    })
worldbank_overview = pd.DataFrame(worldbank_overview_rows)

display(worldbank_overview)


worldbank_gdp.csv


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,,,,,,,...,36926.11307,38213.28924,39378.22841,39754.56007,30408.82709,35207.17091,39083.46164,42783.24891,45428.00273,
1,Africa Eastern and Southern,AFE,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,,,,,,,...,4210.935695,4223.18805,4229.839634,4200.531044,3978.133502,4049.969513,4098.126192,4076.224898,4099.889812,4164.679177
2,Afghanistan,AFG,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,,,,,,,...,2958.785399,2952.998916,2902.392113,2927.245144,2769.685745,2144.16657,1981.710168,1983.81262,1964.452289,
3,Africa Western and Central,AFW,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,,,,,,,...,6034.826989,6009.516114,6026.527572,6070.284769,5698.766702,5702.134886,5818.483224,5886.796636,6009.084662,6139.587294
4,Angola,AGO,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,,,,,,,...,10963.99679,10565.16765,10154.83892,9712.979449,8904.949006,8720.187687,8750.99363,8597.538788,8752.797471,8760.19082


Indicators: ['GDP per capita, PPP (constant 2021 international $)']

worldbank_population_total.csv


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922,55578,56320,57002,57619,58190,...,108727,108735,108908,109203,108587,107700,107310,107359,107995,108785
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728,133534923,137171659,140945536,144904094,149033472,...,623369401,640058741,657801085,675950189,694446100,713090928,731821393,750491370,769280888,788844284
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043,9214083,9404406,9604487,9814318,10036008,...,34700612,35688935,36743039,37856121,39068979,40000412,40578842,41454761,42647492,43844111
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925,99706674,101854756,104089175,106384410,108754449,...,428797338,440150152,451395343,462522286,473687685,484978794,496366058,508318102,520655398,532809933
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654,5301583,5354310,5408320,5464187,5521981,...,29183070,30234839,31297155,32375632,33451132,34532429,35635029,36749906,37885849,39040039


Indicators: ['Population, total']


,File,Rows,Columns,Missing_year_cells_1995_2024
0,worldbank_gdp.csv,265,70,678
1,worldbank_population_total.csv,265,70,30


## 7. Save the small inputs needed by the next two scripts

In [8]:
# Prepared CSVs are saved in dataset_analysis/; raw inputs remain unchanged.
scoped_insurance_rows.to_csv(data_folder / "01_scoped_insurance_history.csv", index=False, encoding="utf-8-sig")
country_labels.to_csv(data_folder / "01_country_labels.csv", index=False, encoding="utf-8-sig")